In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('.')))

from ab_testing import Experiment, required_sample_size
from ab_testing.stats import z_test, welch_t_test, wilson_ci, cuped
from ab_testing.sequential import SequentialTest

In [ ]:
# Constants and config
DATA_DIR = Path('..') / 'data' / 'synthetic'
ALPHA = 0.05
POWER = 0.80
RANDOM_SEED = 42

In [ ]:
# Load synthetic data
users = pd.read_csv(DATA_DIR / 'users.csv', parse_dates=['signup_at', 'churn_at'])
print(f"{len(users):,} users  |  {users['converted'].mean():.1%} conversion rate")

In [ ]:
# --- 1. Sample size planning ---
baseline_rate = users['converted'].mean()
plan = required_sample_size(
    baseline_rate=baseline_rate,
    mde_relative=0.15,
    alpha=ALPHA,
    power=POWER,
    daily_traffic_per_variant=200,
)
print(plan)

In [ ]:
# --- 2. Simulate an A/B test on conversion rate ---
rng = np.random.default_rng(RANDOM_SEED)
n = plan.n_per_variant

control_converted   = rng.binomial(1, baseline_rate, size=n)
treatment_converted = rng.binomial(1, baseline_rate * 1.18, size=n)  # +18% lift

exp = Experiment('checkout_cta_color', metric_type='binary', alpha=ALPHA)
result = exp.analyze(control_converted, treatment_converted)
print(result)

# Wilson CI on each arm
c_lower, c_upper = wilson_ci(control_converted.sum(), n)
t_lower, t_upper = wilson_ci(treatment_converted.sum(), n)
print(f"Control:   {control_converted.mean():.3%}  [{c_lower:.3%}, {c_upper:.3%}]")
print(f"Treatment: {treatment_converted.mean():.3%}  [{t_lower:.3%}, {t_upper:.3%}]")

In [ ]:
# --- 3. Revenue experiment with CUPED ---
n_cuped = 1000
pre_revenue  = rng.lognormal(mean=3.5, sigma=1.2, size=n_cuped * 2)
noise        = rng.normal(0, 0.5, size=n_cuped * 2)
post_revenue = pre_revenue * 0.7 + noise
post_revenue[n_cuped:] += 3.0  # +$3 treatment effect

# Without CUPED
raw_result = welch_t_test(post_revenue[:n_cuped], post_revenue[n_cuped:])
# With CUPED
cuped_result = cuped(
    post_revenue[:n_cuped], post_revenue[n_cuped:],
    pre_revenue[:n_cuped],  pre_revenue[n_cuped:],
)
print('Without CUPED:', raw_result)
print('With CUPED:   ', cuped_result)

In [ ]:
# --- 4. Sequential test simulation ---
seq_test = SequentialTest(alpha=ALPHA)
results = []
for day in range(30):
    c = rng.binomial(1, 0.05, size=100)
    t = rng.binomial(1, 0.06, size=100)
    r = seq_test.update(c, t)
    results.append({'day': day + 1, 'p_value': r.p_value, 'effect': r.effect_size})
    if r.significant:
        print(f'Stopped on day {day + 1}: {r}')
        break

seq_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(seq_df['day'], seq_df['p_value'], label='p-value')
ax.axhline(ALPHA, color='red', linestyle='--', label=f'α = {ALPHA}')
ax.set_xlabel('Day')
ax.set_ylabel('Always-valid p-value')
ax.set_title('Sequential Test: p-value over time')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 5. Summary report ---
summary = exp.summary(control_converted, treatment_converted, result)
summary